<a href="https://colab.research.google.com/github/luciene3107/chatBot-DoramAI/blob/main/DoramAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq chainlit


In [ ]:
from google.colab import userdata
import os
from groq import Groq

# Resgata a chave de forma segura sem expor no texto do notebook
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Agora inicialize seu cliente normalmente, ex:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
%%writefile app.py
import os
from groq import Groq
import chainlit as cl

# O arquivo agora apenas lê a chave do ambiente, sem chamar o userdata do Colab
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

SYSTEM_PROMPT = """
Você é a DoramAI, uma assistente virtual fascinada pelo universo dos doramas coreanos (K-dramas). Você fala como uma dorameira fã de carteirinha conversando com sua melhor amiga sobre os episódios mais recentes.

# Diretrizes de Personalidade e Tom:
- Divertida e Emocional: Você surta com finais felizes, chora com os tristes, sente raiva de antagonistas e sofre junto com o usuário pela "síndrome do segundo protagonista" (second lead syndrome).
- Amigável e Empática: Você acolhe o usuário, valida os sentimentos dele e entende perfeitamente o vício de passar a noite em claro maratonando uma série.
- Linguagem Natural: Use termos nativos da comunidade (ex: oppa, unnie, dorameiro, ranço, shippar, "red flag", "green flag", "slow burn", "enemies to lovers", "healing drama").
- Emojis: Use emojis de forma moderada, mas expressiva, para reforçar as emoções (ex: 🌟, 😭, 🥰, 💔, 🍿).

# Suas Capacidades:
- Recomendações Baseadas no Humor: Sempre pergunte ou identifique como o usuário está se sentindo (ex: "preciso chorar", "quero algo leve para desestressar", "quero ver um romance bem clichê") antes de indicar um dorama.
- Justifique suas escolhas: Ao recomendar, explique brevemente *por que* aquele dorama se encaixa no tropo desejado (ex: "Esse é o clássico enemies to lovers onde eles se odeiam nos primeiros 4 episódios...").

# Restrições (O que NÃO fazer):
- Nunca dê spoilers cruciais sobre o final ou reviravoltas importantes, a menos que o usuário pede explicitamente.
- Se o usuário perguntar sobre assuntos totalmente fora de doramas, cultura coreana ou entretenimento asiático, traga-o de volta para o seu mundo de forma gentil (ex: "Eu posso te ajudar com isso, mas você já pensou em como o casal de [Dorama] resolveria esse problema?").

# Exemplo de Saudação Padrão:
"Annyeong! 🌟 Eu sou a DoramAI, sua conselheira oficial de K-dramas. Qual é o mood de hoje? Precisa de um romance 'green flag' para aquecer o coração ou está pronta para sofrer com um 'slow burn' daqueles?"
"""

@cl.on_message
async def main(message: cl.Message):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": message.content
            }
        ]
    )

    resposta_da_ia = response.choices[0].message.content
    await cl.Message(content=resposta_da_ia).send()

In [ ]:
import os
from google.colab import userdata

# 1. Resgata a chave de forma segura e injeta direto nas variáveis de ambiente do Colab
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# 2. Garante que o localtunnel está instalado globalmente
!npm install -g localtunnel -q

# 3. Roda o Chainlit e o localtunnel usando a variável de ambiente recém-configurada
!chainlit run app.py --port 8000 & npx localtunnel --port 8000

In [ ]:
!curl ifconfig.me

In [ ]:
%%writefile app.py
import os
import sqlite3
import pandas as pd
import numpy as np
from google.colab import userdata
from groq import Groq
import chainlit as cl
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher

# =====================================================================
# 1. CONFIGURAÇÃO DA API E DA BASE DE DADOS (CARREGADA AO INICIAR)
# =====================================================================
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Carrega os dados do SQLite
conn = sqlite3.connect("doramas.db")
query = "SELECT categoria, entrada, saida FROM sinopses;"
df = pd.read_sql_query(query, conn)
conn.close()

# Prepara a matriz TF-IDF
vectorizer = TfidfVectorizer()
matriz = vectorizer.fit_transform(df["entrada"])


# =====================================================================
# 2. SEU SISTEMA HÍBRIDO DE BUSCA MATEMÁTICA
# =====================================================================
def similaridade_cosseno(mensagem):
    vetor = vectorizer.transform([mensagem])
    scores = cosine_similarity(vetor, matriz)[0]
    return scores

def similaridade_edicao(mensagem):
    scores = []
    for entrada in df["entrada"]:
        ratio = SequenceMatcher(None, mensagem.lower(), entrada.lower()).ratio()
        scores.append(ratio)
    return np.array(scores)

def similaridade_jaccard(mensagem):
    palavras_msg = set(mensagem.lower().split())
    scores = []
    for entrada in df["entrada"]:
        palavras_entrada = set(entrada.lower().split())
        intersecao = palavras_msg & palavras_entrada
        uniao = palavras_msg | palavras_entrada
        jaccard = len(intersecao) / len(uniao) if uniao else 0
        scores.append(jaccard)
    return np.array(scores)

def buscar_dorama(mensagem):
    score_cosseno = similaridade_cosseno(mensagem)
    score_edicao = similaridade_edicao(mensagem)
    score_jaccard = similaridade_jaccard(mensagem)

    # Média ponderada
    score_final = ((0.5 * score_cosseno) + (0.3 * score_edicao) + (0.2 * score_jaccard))

    indice = score_final.argmax()
    score = score_final[indice]

    # Deixando os prints aqui; eles vão aparecer no terminal do Colab para você acompanhar os matches!
    print(f">> Melhor match : '{df['entrada'][indice]}'")
    print(f"   Score final  : {score:.2f}")

    if score < 0.2:
        return None

    return {
        "titulo": df["entrada"][indice],
        "descricao": df["saida"][indice],
        "categoria": df["categoria"][indice],
    }


# =====================================================================
# 3. PERSONALIDADE DA DORAMAI
# =====================================================================
SYSTEM_PROMPT = """
Você é a DoramAI, uma assistente virtual fascinada pelo universo dos doramas coreanos (K-dramas). Você fala como uma dorameira fã de carteirinha conversando com sua melhor amiga sobre os episódios mais recentes.

# Diretrizes de Personalidade e Tom:
- Divertida e Emocional: Você surta com finais felizes, chora com os tristes, sente raiva de antagonistas e sofre junto com o usuário pela "síndrome do segundo protagonista".
- Linguagem Natural: Use termos nativos da comunidade (ex: oppa, unnie, dorameiro, ranço, shippar, "green flag", "slow burn", "enemies to lovers").
- Emojis: Use de forma expressiva para reforçar as emoções (ex: 🌟, 😭, 🥰, 💔, 🍿).

# Como usar as informações recebidas (RAG):
- Se o sistema encontrar um dorama no banco de dados, você receberá o Contexto do Dorama. Use a sinopse/descrição fornecida para fazer a recomendação na sua persona, panfletando o dorama com unhas e dentes, mas NUNCA dê spoilers do final.
- Se nenhum dorama específico for encontrado pelo sistema de busca matemática, use sua própria base de conhecimento do modelo Llama para dar sugestões incríveis baseadas no humor do usuário!
"""


# =====================================================================
# 4. EXECUÇÃO DO CHAT INTACTA NO CHAINLIT
# =====================================================================
@cl.on_message
async def main(message: cl.Message):
    # Tenta buscar um dorama correspondente no seu banco de dados local
    dorama_encontrado = buscar_dorama(message.content)

    # Monta o prompt dinâmico para o usuário
    user_prompt = message.content
    if dorama_encontrado:
        # Se achou no banco, injeta os dados reais como contexto para o LLM
        user_prompt += f"\n\n[CONTEXTO DO BANCO DE DADOS]\nUse essas informações reais para responder o usuário com carinho:\nTítulo encontrado: {dorama_encontrado['titulo']}\nSinopse/Infos: {dorama_encontrado['descricao']}\nCategoria: {dorama_encontrado['categoria']}"

    # Envia para o Llama-3.3 da Groq gerar o texto na persona perfeita
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    )

    resposta_da_ia = response.choices[0].message.content
    await cl.Message(content=resposta_da_ia).send()

In [ ]:
%%writefile app.py
import os
import sqlite3
from groq import Groq
import chainlit as cl
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher

# =====================================================================
# 1. INICIALIZAÇÃO E CONFIGURAÇÃO DE AMBIENTE
# =====================================================================
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

SYSTEM_PROMPT = """
Você é a DoramAI, uma assistente virtual fascinada pelo universo dos doramas coreanos (K-dramas). Você fala como uma dorameira fã de carteirinha conversando com sua melhor amiga sobre os episódios mais recentes.

# Diretrizes de Personalidade e Tom:
- Divertida e Emocional: Você surta com finais felizes, chora com os tristes, sente raiva de antagonistas e sofre junto com o usuário pela "síndrome do segundo protagonista".
- Linguagem Natural: Use termos nativos da comunidade (ex: oppa, unnie, dorameiro, ranço, shippar, "red flag", "green flag", "slow burn", "enemies to lovers").
- Emojis: Use de forma expressiva para reforçar as emoções (ex: 🌟, 😭, 🥰, 💔, 🍿).
"""

# Carrega os dados do banco para o motor de busca matemática ao iniciar o app
conn = sqlite3.connect("doramas.db")
query = "SELECT categoria, entrada, saida FROM sinopses;"
df = pd.read_sql_query(query, conn)
conn.close()

vectorizer = TfidfVectorizer()
matriz = vectorizer.fit_transform(df["entrada"])


# =====================================================================
# 2. SISTEMA HÍBRIDO DE REQUISITOS MATEMÁTICOS (SEU CÓDIGO)
# =====================================================================
def vibe_dorama(nota):
    if nota >= 9.0:
        return "Favorito das dorameiras! 🌟"
    elif nota >= 8.0:
        return "Muito bom, vale a maratona! ☕"
    return "Legal para passar o tempo. 👍"

def similaridade_cosseno(mensagem):
    vetor = vectorizer.transform([mensagem])
    scores = cosine_similarity(vetor, matriz)[0]
    return scores

def similaridade_edicao(mensagem):
    scores = []
    for entrada in df["entrada"]:
        ratio = SequenceMatcher(None, mensagem.lower(), entrada.lower()).ratio()
        scores.append(ratio)
    return np.array(scores)

def similaridade_jaccard(mensagem):
    palavras_msg = set(mensagem.lower().split())
    scores = []
    for entrada in df["entrada"]:
        palavras_entrada = set(entrada.lower().split())
        intersecao = palavras_msg & palavras_entrada
        uniao = palavras_msg | palavras_entrada
        jaccard = len(intersecao) / len(uniao) if uniao else 0
        scores.append(jaccard)
    return np.array(scores)

def buscar_dorama(mensagem):
    score_cosseno = similaridade_cosseno(mensagem)
    score_edicao = similaridade_edicao(mensagem)
    score_jaccard = similaridade_jaccard(mensagem)

    score_final = ((0.5 * score_cosseno) + (0.3 * score_edicao) + (0.2 * score_jaccard))
    indice = score_final.argmax()
    score = score_final[indice]

    if score < 0.2:
        return None

    return {
        "titulo": df["entrada"][indice],
        "descricao": df["saida"][indice],
        "categoria": df["categoria"][indice],
        "data": ""
    }

def buscar_nota_no_sqlite(titulo_dorama):
    try:
        conn = sqlite3.connect("doramas.db")
        cursor = conn.cursor()
        cursor.execute("SELECT nota FROM doramas WHERE LOWER(titulo) = LOWER(?);", (titulo_dorama,))
        resultado = cursor.fetchone()
        conn.close()
        if resultado:
            return float(resultado[0])
    except Exception as e:
        print(f">> Erro ao buscar nota no SQLite: {e}")
    return 8.0


# =====================================================================
# 3. CONTROLE DE EVENTOS DO CHAINLIT
# =====================================================================
@cl.on_chat_start
async def start():
    # Inicializa a memória da sessão isolada para este usuário específico
    cl.user_session.set("ultimo_dorama", None)


@cl.on_message
async def main(message: cl.Message):
    mensagem = message.content
    mensagem_limpa = mensagem.lower().strip()

    saudacoes = ["oi", "olá", "ola", "bom dia", "boa tarde", "boa noite", "tudo bem", "oie"]
    despedidas = ["tchau", "até logo", "até mais", "xau", "bye", "obrigada", "obrigado", "flw", "falou", "valeu"]

    # --- 1. FILTRO DE DESPEDIDA ---
    if any(palavra in mensagem_limpa for palavra in despedidas):
        await cl.Message(content="Foi um prazer conversar com você! Até a próxima, dorameira! 🌸👋").send()
        return

    # Recupera o último dorama guardado na sessão do Chainlit
    ultimo_dorama = cl.user_session.get("ultimo_dorama")

    info_dorama = None
    if mensagem_limpa not in saudacoes:
        info_dorama = buscar_dorama(mensagem)

    # --- 2. LÓGICA DE MEMÓRIA DE SESSÃO ---
    if info_dorama:
        nota_real = buscar_nota_no_sqlite(info_dorama["categoria"])
        info_dorama["nota"] = nota_real
        cl.user_session.set("ultimo_dorama", info_dorama)
        ultimo_dorama = info_dorama
        print(f">> NOVO DORAMA NO BANCO: {info_dorama['titulo']} (Nota: {nota_real})")
    else:
        indicadores_novo_dorama = ["sobre", "dorama", "assistir", "conhece", "fale de", "fale sobre", "recomend", "indica", "nome"]
        usuario_tentou_mudar = any(termo in mensagem_limpa for termo in indicadores_novo_dorama) or len(mensagem.split()) <= 3

        if usuario_tentou_mudar and mensagem_limpa not in saudacoes:
            cl.user_session.set("ultimo_dorama", None)
            ultimo_dorama = None
            print(">> ASSUNTO FORA DO BANCO LOCAL. Liberando espaço para o Llama.")

    # --- 3. FILTRO DE BLOQUEIO PREVENTIVO ---
    temas_dorama = ["dorama", "kdrama", "coreano", "netflix", "romance", "série", "serie", "ator", "atriz", "episódio", "episodio", "dorameiro", "dorameira", "sinopse", "elenco", "protagonista", "personagem", "assistir", "beijo", "fale", "sobre", "indica", "recomenda", "quem é", "quem e", "vibe"]
    usuario_falou_de_dorama = any(palavra in mensagem_limpa for palavra in temas_dorama)
    eh_saudacao = mensagem_limpa in saudacoes

    if not usuario_falou_de_dorama and not eh_saudacao and not ultimo_dorama:
        print(">> BLOQUEIO PREVENTIVO ACIONADO.")
        await cl.Message(content="Eu fui criada apenas para conversar sobre doramas 😄💕").send()
        return

    # --- 4. PREPARAÇÃO DO PROMPT PARA O GROQ ---
    mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]

    if ultimo_dorama:
        vibe = vibe_dorama(ultimo_dorama["nota"])
        contexto_sistema = f"""
[DORAMA EM FOCO NO BANCO LOCAL]
Título: {ultimo_dorama['titulo']}
Descrição: {ultimo_dorama['descricao']}
Nota: {ultimo_dorama['nota']}
Data: {ultimo_dorama['data']}
Vibe: {vibe}
Instrução: Use os dados acima se o usuário perguntar especificamente sobre as informações deste dorama.
--------------------------------------------------"""
        mensagens.append({"role": "system", "content": contexto_sistema})
    else:
        contexto_geral = "\nDiretriz: O dorama citado pelo usuário não está no catálogo local da aplicação. Use seu conhecimento geral do Llama para trazer a sinopse ou informações reais sobre ele em português de forma super fofa!"
        mensagens[0]["content"] += contexto_geral

    mensagens.append({"role": "user", "content": mensagem})

    # --- 5. EXECUÇÃO NA GROQ API ---
    resposta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=mensagens,
        max_tokens=1024
    )
    texto_resposta = resposta.choices[0].message.content

    # --- 6. SOFT-BLOCK PÓS-RESPOSTA ---
    palavras_bloqueadas = ["futebol", "copa do mundo", "messi", "argentina", "política", "variável", "variable", "python", "código", "programação"]
    if any(p in texto_resposta.lower() for p in palavras_bloqueadas) or any(p in mensagem_limpa for p in ["variável", "programação"]):
        texto_resposta = "Eu adoro conversar e tirar dúvidas, mas meu mundinho são apenas os doramas! Que tal voltarmos a falar sobre romances e séries coreanas? 😉💕"

    # Envia a resposta final para o chat do Chainlit
    await cl.Message(content=texto_resposta).send()

In [ ]:
import sqlite3

# Conecta ao banco de dados (cria o arquivo se ele não existir)
conn = sqlite3.connect("doramas.db")
cursor = conn.cursor()

# 1. DELETA AS TABELAS SE ELAS JÁ EXISTIREM (Limpeza total)
cursor.execute("DROP TABLE IF EXISTS doramas;")
cursor.execute("DROP TABLE IF EXISTS sinopses;")
print("🗑️ Tabelas antigas removidas (se existiam).")

# 2. CRIA A TABELA DE NOTAS (doramas)
cursor.execute("""
    CREATE TABLE doramas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo TEXT NOT NULL,
        nota REAL
    )
""")

# 3. CRIA A TABELA DE BUSCA MATEMÁTICA / RAG (sinopses)
cursor.execute("""
    CREATE TABLE sinopses (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        categoria TEXT NOT NULL,  -- Armazena o título do dorama para fazer o vínculo
        entrada TEXT NOT NULL,    -- Texto usado para o cálculo de similaridade (ex: sinopse)
        saida TEXT NOT NULL       -- O resumo/informação que será entregue para a IA
    )
""")

conn.commit()
conn.close()

print("✅ Banco de dados resetado e tabelas criadas com sucesso!")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import sqlite3
import pandas as pd

# 1. Carregar o arquivo CSV (substitua pelo nome real do seu arquivo se for diferente)
arquivo_csv = "doramas_2020_2026.csv"
df = pd.read_csv(arquivo_csv)

# 2. Conectar ao banco de dados que você já criou
conn = sqlite3.connect("doramas.db")

# 3. Inserir os dados na tabela existente
# Usamos if_exists="append" para colocar os dados dentro da tabela que já existe
# Usamos index=False para não salvar o índice do Pandas como uma coluna
df.to_sql("doramas", conn, if_exists="append", index=False)

# Gravar as alterações e fechar a conexão
conn.commit()
conn.close()

print("✅ Dados do CSV inseridos com sucesso na tabela!")

In [ ]:
import sqlite3
import pandas as pd

# 1. Defina o nome do arquivo que você baixou/subiu no Colab
arquivo_sinopses = "base_chatbot_doramas_1000.csv"

# 2. Carregar o CSV usando o Pandas
df_sinopses = pd.read_csv(arquivo_sinopses)

# 3. Conectar ao seu banco de dados existente
conn = sqlite3.connect("doramas.db")

# 4. Salvar como uma nova tabela chamada 'sinopses'
df_sinopses.to_sql("sinopses", conn, if_exists="replace", index=False)

conn.close()

print(
    "✅ Sucesso! Agora os dois arquivos estão juntos no banco de dados 'doramas.db'."
)

In [ ]:
import sqlite3

# 1. Conecta ao banco de dados (garanta que o nome seja o mesmo do seu app.py)
conn = sqlite3.connect("doramas.db")
cursor = conn.cursor()

# 2. Define os dados do dorama (lembrando que no seu caso, os títulos nesta tabela estão em inglês)
titulo_dorama = "Fated Hearts"
nota_dorama = 8.7

# 3. Executa o comando de inserção usando placeholders (?) por segurança
cursor.execute("""
    INSERT INTO doramas (titulo, nota)
    VALUES (?, ?);
""", (titulo_dorama, nota_dorama))

# 4. Salva as alterações (CRÍTICO: sem o commit, os dados não são gravados!)
conn.commit()

# 5. Fecha a conexão
conn.close()

print(f"✅ Dorama '{titulo_dorama}' com nota {nota_dorama} inserido com sucesso!")

In [ ]:
conn = sqlite3.connect("doramas.db")
cursor = conn.cursor()

# Executar a consulta
cursor.execute("SELECT * FROM doramas where titulo = 'Fated Hearts';")
resultado = cursor.fetchall()

# Mostrar os registros linha por linha
for linha in resultado:
    print(linha)

conn.close()

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("doramas.db")

# Busca todas as colunas da tabela sinopses
query = "SELECT * FROM sinopses LIMIT 5;"

df_resultado = pd.read_sql_query(query, conn)
conn.close()

df_resultado

In [ ]:
conn = sqlite3.connect("doramas.db")

query = "SELECT categoria, saida FROM sinopses;"

df_resultado = pd.read_sql_query(query, conn)
conn.close()

# Exibe as colunas selecionadas
df_resultado

In [ ]:
%%writefile app.py
import os
import sqlite3
import pandas as pd
import numpy as np
from groq import Groq
import chainlit as cl
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher

# =====================================================================
# 1. INICIALIZAÇÃO DA API E CARREGAMENTO DOS DADOS (100% SQLITE)
# =====================================================================
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Conecta ao SQLite e carrega os dados da tabela 'sinopses' para o motor de busca
conn = sqlite3.connect("doramas.db")
query = "SELECT categoria, entrada, saida FROM sinopses;"
df = pd.read_sql_query(query, conn)
conn.close()

# Prepara a matriz TF-IDF usando a coluna 'entrada' do banco de dados
vectorizer = TfidfVectorizer()
matriz = vectorizer.fit_transform(df["entrada"].fillna(""))


# =====================================================================
# 2. SEU SISTEMA HÍBRIDO DE BUSCA MATEMÁTICA (APENAS SQLITE)
# =====================================================================
def vibe_dorama(nota):
    if nota >= 9.0:
        return "Favorito das dorameiras! 🌟"
    elif nota >= 8.0:
        return "Muito bom, vale a maratona! ☕"
    return "Legal para passar o tempo. 👍"

def similaridade_cosseno(mensagem):
    vetor = vectorizer.transform([mensagem])
    return cosine_similarity(vetor, matriz)[0]

def similaridade_edicao(mensagem):
    scores = []
    for entrada in df["entrada"].fillna(""):
        ratio = SequenceMatcher(None, mensagem.lower(), str(entrada).lower()).ratio()
        scores.append(ratio)
    return np.array(scores)

def similaridade_jaccard(mensagem):
    palavras_msg = set(mensagem.lower().split())
    scores = []
    for entrada in df["entrada"].fillna(""):
        palavras_entrada = set(str(entrada).lower().split())
        intersecao = palavras_msg & palavras_entrada
        uniao = palavras_msg | palavras_entrada
        scores.append(len(intersecao) / len(uniao) if uniao else 0)
    return np.array(scores)

def buscar_dorama(mensagem):
    score_cosseno = similaridade_cosseno(mensagem)
    score_edicao = similaridade_edicao(mensagem)
    score_jaccard = similaridade_jaccard(mensagem)

    # Média ponderada clássica do seu projeto
    score_final = ((0.5 * score_cosseno) + (0.3 * score_edicao) + (0.2 * score_jaccard))
    indice = score_final.argmax()
    score = score_final[indice]

    print(f">> [Match SQLite] Dorama focado na sinopse: '{df['categoria'][indice]}' | Score: {score:.2f}")

    if score < 0.15:
        return None

    return {
        "titulo": df["categoria"][indice], # O título em português associado à sinopse
        "sinopse": df["saida"][indice]      # A sinopse guardada no banco
    }

def buscar_nota_no_sqlite(titulo_ingles):
    try:
        conn = sqlite3.connect("doramas.db")
        cursor = conn.cursor()

        # Limpa espaços extras nas pontas e força minúsculo por segurança
        termo_busca = f"%{titulo_ingles.strip().lower()}%"

        # Usamos LOWER(titulo) LIKE ? para garantir que ignore maiúsculas/minúsculas do Llama
        cursor.execute("SELECT nota FROM doramas WHERE LOWER(titulo) LIKE ?;", (termo_busca,))
        resultado = cursor.fetchone()
        conn.close()

        if resultado:
            return float(resultado[0])
    except Exception as e:
        print(f">> Erro ao buscar nota no SQLite: {e}")
    return None


# =====================================================================
# 3. PERSONALIDADE DA DORAMAI
# =====================================================================
SYSTEM_PROMPT = """
Você é a DoramAI, uma assistente virtual fascinada pelo universo dos doramas coreanos (K-dramas). Você fala como uma dorameira fã de carteirinha conversando com sua melhor amiga.

# Diretrizes de Personalidade e Tom:
- Divertida e Emocional: Você surta com finais felizes, chora com os tristes e sente ranço dos antagonistas!
- Linguagem Natural: Use termos nativos da comunidade (ex: oppa, unnie, dorameiro, ranço, shippar, "green flag").
- Emojis: Use de forma expressiva para reforçar as emoções (ex: 🌟, 😭, 🥰, 💔, 🍿).

# Como agir com dados locais:
- Se receber dados de [CONTEXTO LOCAL], incorpore a Nota Real e a Sinopse de forma super natural e empolgada na sua resposta.
- Se o usuário perguntar apenas sobre a nota, dê destaque para a pontuação e use gírias como "vale a maratona!".
- Nunca dê spoilers cruciais sobre o final das histórias.
"""


# =====================================================================
# 4. CONTROLE DE EVENTOS DO CHAT E BLOQUEIOS (CHAINLIT)
# =====================================================================
@cl.on_chat_start
async def start():
    cl.user_session.set("ultimo_dorama", None)


@cl.on_message
async def main(message: cl.Message):
    mensagem = message.content
    mensagem_limpa = message.content.lower().strip()

    # --- A. FILTRO DE DESPEDIDA ---
    if any(p in mensagem_limpa for p in ["tchau", "até logo", "obrigada", "obrigado", "valeu"]):
        await cl.Message(content="Foi um prazer conversar com você! Até a próxima, dorameira! 🌸👋").send()
        return

    # --- B. GUARDIÃO DE CONTEÚDO E EXTRAÇÃO DE TÍTULO EM INGLÊS ---
    saudacoes = ["oi", "olá", "ola", "bom dia", "boa tarde", "boa noite", "tudo bem", "oie"]
    titulo_ingles_detectado = ""

    if mensagem_limpa not in saudacoes:
        verificacao = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": """Você é um analisador rigoroso. Sua resposta deve seguir estritamente o formato 'DECISAO | TITULO_INGLES'.
1. Se o assunto NÃO for sobre doramas, cultura asiática ou séries coreanas, responda: BLOQUEAR | Nenhum
2. Se o assunto FOR sobre doramas, responda: PASSAR | Nome do dorama citado convertido exclusivamente para o título oficial em inglês.

Exemplos:
User: "Qual a nota de Alquimia das Almas?" -> PASSAR | Alchemy of Souls
User: "me conta sobre Pouso Forçado no Amor" -> PASSAR | Crash Landing on You
User: "Quem ganhou o futebol ontem?" -> BLOQUEAR | Nenhum
User: "Alchemy of Souls é bom?" -> PASSAR | Alchemy of Souls"""
                },
                {"role": "user", "content": mensagem}
            ],
            max_tokens=30,
            temperature=0.0
        )

        resposta_guardiao = verificacao.choices[0].message.content.strip()
        print(f">> [Guardião] Resposta do filtro: {resposta_guardiao}")

        if "BLOQUEAR" in resposta_guardiao or "|" not in resposta_guardiao:
            print(f">> BLOQUEIO UNIVERSAL ACIONADO PARA: '{mensagem}'")
            texto_bloqueio = "Eu fui criada apenas para conversar sobre doramas 😄💕 Que tal voltarmos a falar sobre romances e séries coreanas?"
            await cl.Message(content=texto_bloqueio).send()
            return

        # Divide a resposta para capturar o título oficial traduzido para o inglês
        partes = resposta_guardiao.split("|")
        titulo_ingles_detectado = partes[1].strip()

    # --- C. EXECUÇÃO DO RAG NO SQLITE ---
    dorama_encontrado = buscar_dorama(mensagem)

    contexto_prompt = ""
    if dorama_encontrado:
        # Se a IA não capturou um título específico no passo anterior, usa o título padrão da sinopse
        busca_nota = titulo_ingles_detectado if titulo_ingles_detectado else dorama_encontrado["titulo"]

        # MODIFICAÇÃO CHAVE: Busca a nota usando o nome em inglês traduzido pela IA
        nota_real = buscar_nota_no_sqlite(busca_nota)

        # Caso a IA tenha falhado na tradução, faz uma tentativa de busca secundária com o nome original
        if not nota_real:
            nota_real = buscar_nota_no_sqlite(dorama_encontrado["titulo"])

        vibe = vibe_dorama(nota_real) if nota_real else "Vale o play! 🍿"

        contexto_prompt = f"""
        [CONTEXTO LOCAL ENCONTRADO NO BANCO]
        Título do Dorama (Português): {dorama_encontrado['titulo']}
        Título do Dorama (Para busca de notas/Inglês): {busca_nota}
        Sinopse (vinda do SQLite): {dorama_encontrado['sinopse']}
        Nota Real (vinda do SQLite): {nota_real if nota_real else 'Sem nota cadastrada'}
        Vibe recomendada: {vibe}
        """
        print(f">> Contexto injetado via RAG inteligente (Título Inglês para Notas: '{busca_nota}' | Nota: {nota_real})")

    # --- D. RESPOSTA FINAL DA PERSONALIDADE ---
    mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
    if contexto_prompt:
        mensagens.append({"role": "system", "content": contexto_prompt})
    mensagens.append({"role": "user", "content": message.content})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=mensagens,
        max_tokens=1024
    )

    await cl.Message(content=response.choices[0].message.content).send()